# Data Skew in Spark Join - 

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DataSkewBroadcastJoinDemo")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "true")
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark Version       :", spark.version)
print("Application Name    :", sc.appName)
print("Master              :", sc.master)
print("Default Parallelism :", sc.defaultParallelism)
print("Shuffle Partitions  :", spark.conf.get("spark.sql.shuffle.partitions"))
print("Spark UI            :", sc.uiWebUrl)

In [ ]:
#.config("spark.sql.shuffle.partitions", "4")

In [ ]:
# Partition 0 → 1,000 records
# Partition 1 → 1,100 records
# Partition 2 → 900 records
# Partition 3 → 90,000 records   ← SKEW

In [ ]:
from pyspark.sql import functions as F

# Heavy customer: C001
heavy_customer_df = (
    spark.range(0, 80000)
    .withColumn("transaction_id", F.concat(F.lit("TXN_"), F.col("id")))
    .withColumn("customer_id", F.lit("C001"))
    .withColumn("amount", (F.rand(seed=42) * 1000).cast("double"))
    .select(
        "transaction_id",
        "customer_id",
        "amount"
    )
)

heavy_customer_df.show(5, truncate=False)

In [ ]:
normal_customer_df = (
    spark.range(80000, 100000)
    .withColumn("transaction_id", F.concat(F.lit("TXN_"), F.col("id")))
    .withColumn(
        "customer_id",
        F.concat(
            F.lit("C"),
            F.lpad(
                ((F.col("id") % 20) + 2).cast("string"),
                3,
                "0"
            )
        )
    )
    .withColumn("amount", (F.rand(seed=100) * 1000).cast("double"))
    .select(
        "transaction_id",
        "customer_id",
        "amount"
    )
)

normal_customer_df.show(5, truncate=False)

In [ ]:
transactions_df = heavy_customer_df.union(normal_customer_df)

In [ ]:
transactions_df.count()

In [ ]:
transactions_df \
    .groupBy("customer_id") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(25, truncate=False)